# Kaggle free-P100 inference benchmark

**Execution status: pending Kaggle execution.** This notebook does not install packages, download model weights, or fabricate benchmark results. Run it in a Kaggle notebook with a free Tesla P100 after attaching the final image manifest and the already-prepared model dependencies/weights.

The benchmark measures model-load time separately from steady-state inference throughput, then projects total hours for the full image count across both configured model entries. It uses the same TorchXRayVision preprocessing contract as the planned `inference.py`: grayscale image, TorchXRayVision normalization, center crop, each checkpoint's native resolution, and one-channel tensor input.

## tl;dr

No results are shown yet because this notebook is intentionally unexecuted. After Kaggle execution, the Results cell will print images/second, projected hours per model, combined two-model hours, a 20% throughput-buffered conservative estimate, and the JSON/CSV output locations.

The separate AUROC validation is not part of this benchmark. The blocker is the team-approved evaluation contract: final ground-truth label mapping, fixed split/500-image smoke-test membership, and the exact published-range comparison protocol.

## Context & Methods

### Key assumptions

- The target runtime is Kaggle's free Tesla P100 first; `RUNTIME_TARGET` can be changed later for Colab Pro or CPU.
- The default entries are two distinct public TorchXRayVision architectures: DenseNet-121 (`densenet121-res224-all`) and ResNet-50 (`resnet50-res512-all`). If the team chooses a different second model, replace one entry and declare its native input resolution explicitly.
- The NIH DenseNet weight (`densenet121-res224-nih`) remains supported by `inference.py`, but it has four blank/untrained output slots and is not the default second-backbone benchmark because its raw label vector is not schema-identical to the 18-label `all`/ResNet configuration.
- This notebook excludes model-download and disk-cache time from steady-state throughput. It reports model-load time separately.
- The conservative estimate is an explicit 20% throughput reduction from the observed mean; change `CONSERVATIVE_THROUGHPUT_FACTOR` if the team chooses another buffer.

Official API references: [TorchXRayVision README](https://github.com/mlmed/torchxrayvision) and [TorchXRayVision documentation](https://mlmed.org/torchxrayvision/).

## 1. Visible parameters and assumptions

Edit this cell in Kaggle. Paths are placeholders because the final manifest and model artifacts are not in this repository. No package installation or weight download is performed here.

In [ ]:
from pathlib import Path
import json
import math
import os
import platform
import sys
import time

# ---- Input and output paths ----
IMAGE_MANIFEST_PATH = "/kaggle/input/REPLACE_WITH_DATASET/image_manifest.csv"
MANIFEST_PATH_COLUMN = "image_path"
IMAGE_INDEX_COLUMN = "Image Index"
TOTAL_IMAGE_COUNT = 0  # Set to the authoritative full-dataset count; 0 means infer from the manifest.
OUTPUT_DIR = "/kaggle/working/inference_benchmark"
SUMMARY_JSON_NAME = "benchmark_summary.json"
SUMMARY_CSV_NAME = "benchmark_summary.csv"

# ---- Runtime and benchmark controls ----
RUNTIME_TARGET = "kaggle-free-p100"  # kaggle-free-p100, colab-pro, or cpu
REQUESTED_DEVICE = "cuda"  # cuda or cpu; unavailable CUDA falls back to CPU with a visible warning.
BATCH_SIZE = 64
NUM_WORKERS = 2
PIN_MEMORY = True
NUM_WARMUP_BATCHES = 5
NUM_TIMED_BATCHES = 50
TIMED_IMAGES = 0  # 0 means up to NUM_TIMED_BATCHES * BATCH_SIZE, bounded by manifest size.
PRECISION_MODE = "fp32"  # fp32, amp_fp16, or amp_bf16. P100 supports fp16 AMP; use fp32 first if comparing exactly.
CONSERVATIVE_THROUGHPUT_FACTOR = 0.80
STRICT_SCHEMA_COMPATIBILITY = True

# Two provisional distinct TorchXRayVision backbones. Replace these only if the team
# selects a different official second architecture; preserve each model's native resolution.
BACKBONE_CONFIGS = [
    {
        "name": "densenet121-res224-all",
        "loader": "torchxrayvision.DenseNet",
        "weights": "densenet121-res224-all",
        "input_resolution": 224,
        "outputs_are_sigmoid_scores": True,
    },
    {
        "name": "resnet50-res512-all",
        "loader": "torchxrayvision.ResNet",
        "weights": "resnet50-res512-all",
        "input_resolution": 512,
        "outputs_are_sigmoid_scores": True,
    },
]

# Optional later adapter hook. Leave None to use the official TorchXRayVision path below.
INFERENCE_PY_PATH = None  # e.g. "/kaggle/working/inference.py" after its public loader contract is finalized.

if len(BACKBONE_CONFIGS) != 2:
    raise ValueError("Configure exactly two model entries for the two-backbone projection.")
if not 0 < CONSERVATIVE_THROUGHPUT_FACTOR <= 1:
    raise ValueError("CONSERVATIVE_THROUGHPUT_FACTOR must be in (0, 1].")
print(json.dumps({
    "runtime_target": RUNTIME_TARGET,
    "manifest": IMAGE_MANIFEST_PATH,
    "resolutions": {cfg["name"]: cfg["input_resolution"] for cfg in BACKBONE_CONFIGS},
    "batch_size": BATCH_SIZE,
    "warmup_batches": NUM_WARMUP_BATCHES,
    "timed_batches": NUM_TIMED_BATCHES,
    "precision": PRECISION_MODE,
    "models": [cfg["name"] for cfg in BACKBONE_CONFIGS],
}, indent=2))

## 2. Kaggle execution note and imports

Kaggle setup should provide PyTorch, torchvision, Pillow, pandas, NumPy, and TorchXRayVision. This notebook deliberately does not run `pip install`, download a checkpoint, or assume a previous Kaggle session. If a dependency or weight is missing, stop and fix the Kaggle environment before treating any timing as valid.

In [ ]:
import importlib.util
from contextlib import nullcontext

try:
    import numpy as np
    import pandas as pd
    import torch
    from PIL import Image
    import torchvision
    import torchxrayvision as xrv
except ImportError as exc:
    raise RuntimeError(
        "Kaggle dependencies are unavailable. Attach/use the intended Kaggle environment; this notebook does not install them."
    ) from exc

if REQUESTED_DEVICE == "cuda" and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif REQUESTED_DEVICE == "cuda":
    DEVICE = torch.device("cpu")
    print("WARNING: CUDA was requested but is unavailable; timing will be CPU timing.")
else:
    DEVICE = torch.device("cpu")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    DEVICE_NAME = torch.cuda.get_device_name(DEVICE)
else:
    DEVICE_NAME = platform.processor() or platform.machine() or "CPU"

if PRECISION_MODE not in {"fp32", "amp_fp16", "amp_bf16"}:
    raise ValueError("PRECISION_MODE must be fp32, amp_fp16, or amp_bf16.")
if PRECISION_MODE == "amp_bf16" and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("amp_bf16 was selected but this GPU does not report bfloat16 support.")

print(f"Python={sys.version.split()[0]} | PyTorch={torch.__version__} | device={DEVICE} | device_name={DEVICE_NAME}")
if RUNTIME_TARGET == "kaggle-free-p100" and DEVICE.type == "cuda" and "P100" not in DEVICE_NAME.upper():
    print(f"WARNING: target is Kaggle free-P100 but detected {DEVICE_NAME!r}; record this before comparing throughput.")


## 3. Load and validate the image manifest

Supported manifest formats are CSV/TSV, Parquet, JSON records, JSONL, and one-path-per-line text. The standardized dataframe must contain a unique `Image Index` plus a resolved `image_path`. Relative paths are resolved relative to the manifest's directory.

In [ ]:
def load_image_manifest(manifest_path: str, path_column: str, image_index_column: str) -> pd.DataFrame:
    path = Path(manifest_path)
    if not path.exists():
        raise FileNotFoundError(f"Image manifest not found: {path}")
    suffix = path.suffix.lower()
    if suffix in {".txt", ".list", ".lst"}:
        values = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        frame = pd.DataFrame({path_column: values})
    elif suffix == ".jsonl":
        frame = pd.read_json(path, lines=True)
    elif suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict) and "images" in payload:
            payload = payload["images"]
        frame = pd.DataFrame(payload if isinstance(payload, list) else [payload])
    elif suffix in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)
    elif suffix in {".tsv", ".tab"}:
        frame = pd.read_csv(path, sep="\t")
    else:
        frame = pd.read_csv(path)
    if path_column not in frame.columns:
        raise KeyError(f"Manifest must contain path column {path_column!r}; found {list(frame.columns)}")
    frame = frame.copy()
    frame[path_column] = frame[path_column].astype(str)
    frame["image_path"] = frame[path_column].map(lambda value: str((path.parent / value).resolve()) if not Path(value).is_absolute() else value)
    if image_index_column not in frame.columns:
        frame[image_index_column] = frame["image_path"].map(lambda value: Path(value).name)
    frame[image_index_column] = frame[image_index_column].astype(str)
    if frame[image_index_column].duplicated().any():
        raise ValueError(f"{image_index_column!r} must be unique; duplicate count={int(frame[image_index_column].duplicated().sum())}")
    return frame.reset_index(drop=True)

manifest = load_image_manifest(IMAGE_MANIFEST_PATH, MANIFEST_PATH_COLUMN, IMAGE_INDEX_COLUMN)
if manifest.empty:
    raise ValueError("Image manifest is empty; refusing to benchmark zero images.")
if TOTAL_IMAGE_COUNT == 0:
    TOTAL_IMAGE_COUNT = len(manifest)
if TOTAL_IMAGE_COUNT < len(manifest):
    raise ValueError(f"TOTAL_IMAGE_COUNT={TOTAL_IMAGE_COUNT} is smaller than manifest rows={len(manifest)}.")
missing_paths = [p for p in manifest["image_path"].head(1000) if not Path(p).exists()]
if missing_paths:
    raise FileNotFoundError(f"At least one manifest image path is missing; first examples: {missing_paths[:3]}")
print(f"Manifest rows available: {len(manifest):,}; projected full image count: {TOTAL_IMAGE_COUNT:,}")
print(manifest[[IMAGE_INDEX_COLUMN, "image_path"]].head(3).to_string(index=False))

## 4. Model loading, preprocessing, and schema checks

This is the critical compatibility gate. It checks that every configured model declares its native input resolution, exposes a non-empty pathology-name list, emits two-dimensional scores, and has exactly the same ordered output schema as the first model. A mismatch stops the run instead of allowing silent label misalignment.

In [ ]:
def validate_model_config(config: dict) -> None:
    required = {"name", "loader", "weights", "input_resolution", "outputs_are_sigmoid_scores"}
    missing = required.difference(config)
    if missing:
        raise KeyError(f"Model config {config.get('name', '<unnamed>')} missing {sorted(missing)}")
    if not isinstance(config["input_resolution"], int) or config["input_resolution"] <= 0:
        raise ValueError(f"Model {config['name']} must declare a positive native input resolution.")

def make_preprocess(resolution: int):
    transform = torchvision.transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(resolution),
    ])
    def preprocess(path: str) -> torch.Tensor:
        image = np.asarray(Image.open(path).convert("L"))
        image = xrv.datasets.normalize(image, 255)
        image = image[None, ...]
        image = transform(image)
        tensor = torch.from_numpy(np.asarray(image)).float()
        if tuple(tensor.shape) != (1, resolution, resolution):
            raise ValueError(f"Preprocessing produced {tuple(tensor.shape)} for {path}; expected (1, {resolution}, {resolution}).")
        return tensor
    return preprocess

def load_model(config: dict):
    validate_model_config(config)
    if INFERENCE_PY_PATH is not None:
        raise NotImplementedError(
            "INFERENCE_PY_PATH is set, but its loader API is not assumed. Add the finalized inference.py adapter here and keep the output contract below."
        )
    if config["loader"] == "torchxrayvision.DenseNet":
        model = xrv.models.DenseNet(weights=config["weights"])
    elif config["loader"] == "torchxrayvision.ResNet":
        model = xrv.models.ResNet(weights=config["weights"])
    else:
        raise ValueError(f"Unsupported loader {config['loader']!r}; add an explicit adapter.")
    model = model.to(DEVICE).eval()
    pathology_names = list(getattr(model, "pathologies", getattr(model, "targets", [])))
    if not pathology_names:
        raise ValueError(f"{config['name']} exposes no pathology/target names; refusing to infer unlabeled columns.")
    return model, pathology_names

for cfg in BACKBONE_CONFIGS:
    validate_model_config(cfg)
print("Resolution checks passed: every configured model declares its native input resolution; no silent 224 resize is applied.")

class ImagePathDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame, preprocess):
        self.frame = frame
        self.preprocess = preprocess
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.preprocess(row["image_path"]), row[IMAGE_INDEX_COLUMN]

def autocast_context():
    if DEVICE.type != "cuda" or PRECISION_MODE == "fp32":
        return nullcontext()
    dtype = torch.float16 if PRECISION_MODE == "amp_fp16" else torch.bfloat16
    return torch.autocast(device_type="cuda", dtype=dtype)

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def model_scores(model, images, outputs_are_sigmoid_scores: bool):
    with autocast_context():
        outputs = model(images)
    if outputs.ndim != 2:
        raise ValueError(f"Model output must be [batch, labels], got {tuple(outputs.shape)}")
    return outputs if outputs_are_sigmoid_scores else torch.sigmoid(outputs)

## 5. Benchmark load time and steady-state throughput

Warmup batches are excluded from timing. Timed throughput covers host-to-device transfer plus model forward for already-decoded batches; image decoding and preprocessing are deliberately not mixed into the model steady-state number. Re-run with a separate end-to-end loader benchmark if disk/decode throughput becomes the bottleneck.

In [ ]:
def make_loader(preprocess):
    dataset = ImagePathDataset(manifest, preprocess)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and DEVICE.type == "cuda",
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )

def benchmark_one(config: dict, expected_schema=None):
    load_start = time.perf_counter()
    model, pathology_names = load_model(config)
    observed_weight_fields = {}
    for attribute in ("weights", "weight_name", "weight_url", "weight_path"):
        if hasattr(model, attribute):
            value = getattr(model, attribute)
            if isinstance(value, (str, int, float)):
                observed_weight_fields[attribute] = value
    observed_weight_text = json.dumps(observed_weight_fields, default=str)
    weights_match_observed_model = None if not observed_weight_fields else config["weights"] in observed_weight_text
    if weights_match_observed_model is False:
        raise ValueError(
            f"Loaded model weight marker does not match intended {config['weights']!r}: {observed_weight_fields}"
        )
    if weights_match_observed_model is None:
        print(f"WARNING: {config['name']} exposes no inspectable weight marker; provenance is recorded from the explicit loader argument.")
    load_seconds = time.perf_counter() - load_start
    if expected_schema is not None and pathology_names != expected_schema:
        message = (
            f"Output schema mismatch for {config['name']}.\n"
            f"Expected ordered labels: {expected_schema}\n"
            f"Observed ordered labels: {pathology_names}"
        )
        if STRICT_SCHEMA_COMPATIBILITY:
            raise ValueError(message)
        print("WARNING: " + message)
    preprocess = make_preprocess(config["input_resolution"])
    loader = make_loader(preprocess)
    iterator = iter(loader)
    with torch.inference_mode():
        for _ in range(NUM_WARMUP_BATCHES):
            try:
                images, _ = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                images, _ = next(iterator)
            images = images.to(DEVICE, non_blocking=True)
            _ = model_scores(model, images, config["outputs_are_sigmoid_scores"])
        synchronize()
        timed_images = 0
        timed_batches = 0
        synchronize()
        timer_start = time.perf_counter()
        while timed_batches < NUM_TIMED_BATCHES and (TIMED_IMAGES == 0 or timed_images < TIMED_IMAGES):
            try:
                images, _ = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                images, _ = next(iterator)
            if TIMED_IMAGES > 0 and timed_images + len(images) > TIMED_IMAGES:
                images = images[: TIMED_IMAGES - timed_images]
            images = images.to(DEVICE, non_blocking=True)
            _ = model_scores(model, images, config["outputs_are_sigmoid_scores"])
            timed_images += len(images)
            timed_batches += 1
        synchronize()
        inference_seconds = time.perf_counter() - timer_start
    if timed_images <= 0 or inference_seconds <= 0:
        raise RuntimeError("No positive timed sample was collected.")
    images_per_second = timed_images / inference_seconds
    projected_hours = TOTAL_IMAGE_COUNT / images_per_second / 3600.0
    conservative_ips = images_per_second * CONSERVATIVE_THROUGHPUT_FACTOR
    conservative_hours = TOTAL_IMAGE_COUNT / conservative_ips / 3600.0
    return {
        "model": config["name"],
        "configured_loader": config["loader"],
        "configured_weights": config["weights"],
        "input_resolution": config["input_resolution"],
        "pathology_names": pathology_names,
        "model_class": model.__class__.__name__,
        "observed_weight_fields": observed_weight_fields,
        "weights_match_observed_model": weights_match_observed_model,
        "model_load_seconds": load_seconds,
        "timed_batches": timed_batches,
        "timed_images": timed_images,
        "steady_state_inference_seconds": inference_seconds,
        "images_per_second": images_per_second,
        "projected_full_dataset_hours": projected_hours,
        "conservative_images_per_second": conservative_ips,
        "conservative_full_dataset_hours": conservative_hours,
    }

results = []
reference_schema = None
for index, config in enumerate(BACKBONE_CONFIGS, start=1):
    print(f"[{index}/{len(BACKBONE_CONFIGS)}] Benchmarking {config['name']} ...")
    result = benchmark_one(config, expected_schema=reference_schema)
    if reference_schema is None:
        reference_schema = result["pathology_names"]
    results.append(result)
    print(f"  {result['images_per_second']:.2f} images/s | {result['projected_full_dataset_hours']:.2f} h projected | load {result['model_load_seconds']:.1f} s")

## 6. Results, conservative projection, and output files

The combined estimate is the sum of the two sequential full-dataset projections. It does not assume both backbones can share GPU memory or run concurrently. The notebook writes compact JSON and CSV summaries to the configurable output folder.

In [ ]:
summary_frame = pd.DataFrame([{
    **{key: value for key, value in result.items() if key != "pathology_names"},
    "pathology_schema": "|".join(result["pathology_names"]),
} for result in results])
combined_hours = float(summary_frame["projected_full_dataset_hours"].sum())
combined_conservative_hours = float(summary_frame["conservative_full_dataset_hours"].sum())
summary_metadata = {
    "status": "executed_in_kaggle",
    "runtime_target": RUNTIME_TARGET,
    "device": str(DEVICE),
    "device_name": DEVICE_NAME,
    "python_version": sys.version.split()[0],
    "pytorch_version": torch.__version__,
    "manifest_path": IMAGE_MANIFEST_PATH,
    "manifest_rows_available": int(len(manifest)),
    "projected_total_image_count": int(TOTAL_IMAGE_COUNT),
    "batch_size": int(BATCH_SIZE),
    "warmup_batches": int(NUM_WARMUP_BATCHES),
    "timed_batches_per_model": int(NUM_TIMED_BATCHES),
    "precision_mode": PRECISION_MODE,
    "benchmark_resolutions": {result["model"]: int(result["input_resolution"]) for result in results},
    "schema_compatible": bool(all(result["pathology_names"] == reference_schema for result in results)),
    "combined_projected_hours": combined_hours,
    "combined_conservative_hours": combined_conservative_hours,
    "conservative_throughput_factor": CONSERVATIVE_THROUGHPUT_FACTOR,
    "preprocessing_note": "Timed loop includes steady-state DataLoader image decode/preprocessing at each model's configured native resolution and model forward; model loading and warm-up are reported separately and excluded from throughput.",
    "auroc_status": "not_run; requires team-approved labels, split, and published-range protocol",
}
if not summary_metadata["schema_compatible"]:
    raise ValueError("Schema compatibility check failed; do not use these results downstream.")

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
summary_payload = {"metadata": summary_metadata, "models": results}
json_path = output_dir / SUMMARY_JSON_NAME
csv_path = output_dir / SUMMARY_CSV_NAME
json_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
summary_frame.to_csv(csv_path, index=False)

display_columns = [
    "model", "images_per_second", "projected_full_dataset_hours",
    "conservative_full_dataset_hours", "model_load_seconds", "timed_images",
]
print(summary_frame[display_columns].to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print(f"\nCombined projected hours (sequential): {combined_hours:.3f}")
print(f"Combined conservative hours: {combined_conservative_hours:.3f}")
print(f"JSON summary: {json_path}")
print(f"CSV summary:  {csv_path}")

## 7. Interpretation and next steps

- Do not claim any throughput number until this notebook has run on the intended Kaggle GPU and the device name has been recorded.
- If the conservative full-dataset estimate is too long, benchmark a clearly documented subset first. Do not switch to a lower input resolution as a free optimization: these pretrained models are resolution-specific, and resizing a 224-resolution checkpoint to another size changes the model's intended preprocessing contract.
- If the team selects TorchXRayVision `resnet50-res512-all`, set its explicit native resolution to 512 and update the preprocessing/resolution check; do not label a 512-native model as a 224 benchmark.
- After `inference.py` is finalized, add its loader as an explicit adapter and compare one batch's scores, ordered pathology names, and preprocessing shape against this notebook before using the projections.
- Run AUROC only after the team confirms the exact label mapping, split, and comparison protocol. That is the specific blocker for the later AUROC claim; it is independent of this runtime benchmark.